# Fire-Spread Explorer

Simulate wildfire propagation with **pyretechnics**, fed by **Google Earth Engine** data (US).

All logic lives in the `firesim` package. **You only edit the scenario cell below** — set the
area of interest, ignition point, ignition date, and projection period, then run the notebook.

| Layer | Source |
| --- | --- |
| Slope / aspect | `USGS/SRTMGL1_003` (SRTM DEM) |
| Fuel model + water mask | LANDFIRE 2023 `FBFM40` (Scott & Burgan 40), `fuel_source="landfire"` |
| Wind, temperature, 100 hr dead FM | `IDAHO_EPSCOR/GRIDMET` |
| 1 hr / 10 hr dead FM | Simard EMC from GRIDMET |
| Canopy cover / height / base height / bulk density | LANDFIRE 2023 `CC` / `CH` / `CBH` / `CBD` (enables crown fire) |
| Live fuel moisture | seasonal constants |

> MVP: live moisture as constants, historical weather via GRIDMET. LANDFIRE fuels cover CONUS
> only; set `fuel_source="nlcd"` for the old NLCD crosswalk (surface fire only).
> See `docs/pyretechnics_weathernext3_inputs.md` for the full source analysis and upgrade paths.

In [ ]:
# Make the firesim package importable regardless of the notebook's working directory.
import pathlib
import sys

root = pathlib.Path.cwd()
if not (root / "firesim").exists():
    root = root.parent
sys.path.insert(0, str(root))

from firesim import SimulationConfig, run, viz

## 1. Define your scenario

Edit the values below. `aoi_bounds` is `(west, south, east, north)` in lon/lat, and the
ignition point must fall inside it. Use a **historical** date (GRIDMET coverage, 1979-present).

In [ ]:
config = SimulationConfig(
    aoi_bounds=(-120.55, 39.00, -120.30, 39.20),  # (west, south, east, north)
    ignition_lonlat=(-120.45, 39.10),             # (lon, lat) fire departure
    ignition_date="2024-08-15",                   # YYYY-MM-DD (historical)
    projection_days=7,                             # projection horizon
    weather_source="gridmet",                      # this notebook uses GRIDMET
    fuel_source="landfire",                       # "landfire" (LANDFIRE 2023) | "nlcd" (crosswalk)
    enable_crown_fire=True,                        # False = surface fire only
)
config

## 2. Run the simulation

Fetches all Earth Engine layers and runs the spread engine. The first run triggers an Earth
Engine authentication prompt for your `EE_PROJECT` (set in `.env`).

In [ ]:
results = run(config)
results["stats"]

## 3. Explore the fire on an interactive map

Toggle the layers in the control (top-right): arrival-day overlay, **fireline intensity (kW/m)**,
**flame-length severity class**, and the daily perimeters.


In [ ]:
viz.build_map(config, results)

## 4. Charts

Cumulative burned area over time, flame-length distribution, and spread rate over time.

In [ ]:
viz.plot_charts(config, results);